# Notebook 00 — Setup SQL Server (SeguroDB)

Este notebook cria o banco de dados **SeguroDB** no SQL Server 2025, define as 11 tabelas do domínio de seguros de automóvel e carrega os dados de seed a partir dos CSVs da pasta `/data`.

**Pré-requisito:** `docker-compose up -d` deve estar rodando.

## 1. Configuração e conexão

In [ ]:
import os
import pyodbc
import pandas as pd
from dotenv import load_dotenv
import time

load_dotenv()

DB_SERVER = os.getenv('DB_SERVER', 'localhost')
DB_PORT   = os.getenv('DB_PORT', '1433')
DB_USER   = os.getenv('DB_USER', 'sa')
DB_PASS   = os.getenv('DB_PASSWORD', 'Sa@123456')
DB_NAME   = os.getenv('DB_DATABASE', 'SeguroDB')

CONN_MASTER = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={DB_SERVER},{DB_PORT};"
    f"DATABASE=master;"
    f"UID={DB_USER};"
    f"PWD={DB_PASS};"
    f"TrustServerCertificate=yes;"
)

CONN_DB = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={DB_SERVER},{DB_PORT};"
    f"DATABASE={DB_NAME};"
    f"UID={DB_USER};"
    f"PWD={DB_PASS};"
    f"TrustServerCertificate=yes;"
)

print(f'Conectando em: {DB_SERVER}:{DB_PORT}')

## 2. Criar banco de dados SeguroDB

In [ ]:
conn = pyodbc.connect(CONN_MASTER, autocommit=True)
cursor = conn.cursor()

cursor.execute(f"""
IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = '{DB_NAME}')
    CREATE DATABASE {DB_NAME};
""")

print(f'Banco {DB_NAME} criado/verificado com sucesso!')
cursor.close()
conn.close()

## 3. Criar tabelas (DDL)

In [ ]:
conn = pyodbc.connect(CONN_DB, autocommit=True)
cursor = conn.cursor()

ddl_statements = [
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name='regiao')
    CREATE TABLE regiao (
        id    INT PRIMARY KEY,
        nome  NVARCHAR(50) NOT NULL,
        sigla CHAR(2) NOT NULL
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name='estado')
    CREATE TABLE estado (
        id        INT PRIMARY KEY,
        nome      NVARCHAR(50) NOT NULL,
        sigla     CHAR(2) NOT NULL,
        regiao_id INT NOT NULL REFERENCES regiao(id)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name='municipio')
    CREATE TABLE municipio (
        id          INT PRIMARY KEY,
        nome        NVARCHAR(100) NOT NULL,
        estado_id   INT NOT NULL REFERENCES estado(id),
        cep_prefixo CHAR(5)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name='marca')
    CREATE TABLE marca (
        id           INT PRIMARY KEY,
        nome         NVARCHAR(50) NOT NULL,
        pais_origem  NVARCHAR(50)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name='modelo')
    CREATE TABLE modelo (
        id              INT PRIMARY KEY,
        nome            NVARCHAR(50) NOT NULL,
        marca_id        INT NOT NULL REFERENCES marca(id),
        categoria       NVARCHAR(20),
        ano_lancamento  INT
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name='cliente')
    CREATE TABLE cliente (
        id               INT PRIMARY KEY,
        nome             NVARCHAR(100) NOT NULL,
        cpf              CHAR(14) UNIQUE NOT NULL,
        data_nascimento  DATE,
        email            NVARCHAR(100),
        sexo             CHAR(1)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name='endereco')
    CREATE TABLE endereco (
        id           INT PRIMARY KEY,
        cliente_id   INT NOT NULL REFERENCES cliente(id),
        municipio_id INT NOT NULL REFERENCES municipio(id),
        logradouro   NVARCHAR(150) NOT NULL,
        numero       NVARCHAR(10),
        complemento  NVARCHAR(50),
        cep          CHAR(9)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name='telefone')
    CREATE TABLE telefone (
        id         INT PRIMARY KEY,
        cliente_id INT NOT NULL REFERENCES cliente(id),
        tipo       NVARCHAR(20),
        numero     NVARCHAR(20) NOT NULL
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name='carro')
    CREATE TABLE carro (
        id              INT PRIMARY KEY,
        cliente_id      INT NOT NULL REFERENCES cliente(id),
        modelo_id       INT NOT NULL REFERENCES modelo(id),
        ano_fabricacao  INT,
        ano_modelo      INT,
        placa           CHAR(8) UNIQUE,
        cor             NVARCHAR(30),
        renavam         CHAR(11) UNIQUE,
        valor_fipe      DECIMAL(12,2)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name='apolice')
    CREATE TABLE apolice (
        id               INT PRIMARY KEY,
        cliente_id       INT NOT NULL REFERENCES cliente(id),
        carro_id         INT NOT NULL REFERENCES carro(id),
        numero_apolice   NVARCHAR(20) UNIQUE NOT NULL,
        data_inicio      DATE NOT NULL,
        data_fim         DATE NOT NULL,
        tipo_cobertura   NVARCHAR(20),
        valor_premio     DECIMAL(10,2),
        valor_cobertura  DECIMAL(12,2),
        status           NVARCHAR(10) DEFAULT 'ativa'
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name='sinistro')
    CREATE TABLE sinistro (
        id                INT PRIMARY KEY,
        apolice_id        INT NOT NULL REFERENCES apolice(id),
        data_ocorrencia   DATE NOT NULL,
        tipo_sinistro     NVARCHAR(20),
        descricao         NVARCHAR(500),
        valor_dano        DECIMAL(12,2),
        valor_indenizacao DECIMAL(12,2),
        status            NVARCHAR(15) DEFAULT 'em_analise'
    );
    """
]

for stmt in ddl_statements:
    cursor.execute(stmt)

print('11 tabelas criadas com sucesso!')
cursor.close()
conn.close()

## 4. Carregar dados dos CSVs

In [ ]:
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')
# Se rodar o notebook de dentro da pasta notebooks/, subimos um nível
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), '..', 'data')

# Ordem de carga respeitando as FKs
TABELAS = [
    'regiao', 'estado', 'municipio',
    'marca', 'modelo',
    'cliente', 'endereco', 'telefone',
    'carro', 'apolice', 'sinistro'
]

def insert_dataframe(cursor, tabela, df, batch_size=2000):
    cols = ', '.join(df.columns)
    placeholders = ', '.join(['?' for _ in df.columns])
    sql = f"INSERT INTO {tabela} ({cols}) VALUES ({placeholders})"
    rows = [tuple(row) for _, row in df.iterrows()]
    for i in range(0, len(rows), batch_size):
        cursor.executemany(sql, rows[i:i+batch_size])
    return len(rows)

conn = pyodbc.connect(CONN_DB, autocommit=True)
cursor = conn.cursor()

total_registros = 0
for tabela in TABELAS:
    csv_path = os.path.join(DATA_DIR, f'{tabela}.csv')
    df = pd.read_csv(csv_path)

    # Limpa a tabela antes de inserir (para reexecuções)
    cursor.execute(f"DELETE FROM {tabela}")

    count = insert_dataframe(cursor, tabela, df)
    total_registros += count
    print(f'  {tabela:15s} → {count:4d} registros')

cursor.close()
conn.close()
print(f'\nTotal carregado: {total_registros} registros em {len(TABELAS)} tabelas')

## 5. Validação

In [ ]:
conn = pyodbc.connect(CONN_DB)
cursor = conn.cursor()

print('=== Contagem de registros por tabela ===')
for tabela in TABELAS:
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]
    print(f'  {tabela:15s}: {count:5d} registros')

cursor.close()
conn.close()